In [3]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from matplotlib import pyplot

# check which device to use
print(f'cuda {torch.cuda.is_available()}, mps: {torch.backends.mps.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'device: {DEVICE}')



cuda False, mps: True
device: mps


In [ ]:
def visualizeDataItem(dataItem):
    (sampleImageAt0, labelAt0) = dataItem
    print(f'Data item label: {labelAt0}')
    pyplot.imshow(sampleImageAt0)
    pyplot.show()

# Load MNIST dataset
mnist_train = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), target_transform=None, download=False)
mnist_test = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), target_transform=None, download=False)

print(f'Mnist Train Dataset Size: {len(mnist_train)}')
print(f'Mnist Test Dataset Size: {len(mnist_test)}')

# print a sample data item
# print(f'Train Dataitem at index 0: {mnist_train.__getitem__(0)}')
# print(f'Test Dataitem at index 0: {mnist_test.__getitem__(0)}')

# visualize the data items
# visualizeDataItem(mnist_train.__getitem__(0))
# visualizeDataItem(mnist_test.__getitem__(0))


100%|██████████| 9.91M/9.91M [00:21<00:00, 463kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 115kB/s]
100%|██████████| 1.65M/1.65M [00:11<00:00, 145kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.70MB/s]

Mnist Train Dataset Size: 60000
Mnist Test Dataset Size: 10000


### DataLoader

In [5]:
# Create a DataLoader
BATCH_SIZE = 64
train_loader = DataLoader(dataset=mnist_train, batch_size=BATCH_SIZE)
test_loader = DataLoader(dataset=mnist_test, batch_size=BATCH_SIZE)

# Loop over the items in the dataloader and print the first item's shape
# for item in train_loader:
#     print(f'item: {item}')
#     break

for image, label in train_loader:
    print(f'image: {image.shape}, label: {label.shape}')
    break

image: torch.Size([64, 1, 28, 28]), label: torch.Size([64])


In [6]:
# Create the model (that would classify images into its corresponding labels) architecture

INPUT_LAYER_SIZE = 28 * 28 # flattened 28x28 image = 784
HIDDEN_LAYER_SIZE = 128
OUTPUT_LAYER_SIZE = 10

seq_model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=INPUT_LAYER_SIZE, out_features=HIDDEN_LAYER_SIZE),
        nn.ReLU(),
        nn.Linear(in_features=HIDDEN_LAYER_SIZE, out_features=HIDDEN_LAYER_SIZE),
        nn.ReLU(),
        nn.Linear(in_features=HIDDEN_LAYER_SIZE, out_features=OUTPUT_LAYER_SIZE)
    ).to(device=DEVICE)


In [7]:
# Train the model

from tqdm.auto import tqdm
from torch.nn import functional as F
from torch import Tensor
from torch.optim import Adam

def train_one_epoch(model: nn.Sequential, optimizer: Adam, data_loader: DataLoader):
    
    # put the model in training mode
    model.train()

    total_loss = 0.0
    total_samples_processed = 0
    correct_predictions = 0

    # iterate over (mini)batches of data from the data loader and train the model for this 1 epoch
    for x, label in tqdm(iterable=data_loader):

        # print(f'x shape: {x.shape}, label shape: {label.shape}')

        # move the data to device
        x, label = x.to(DEVICE), label.to(DEVICE)

        # Infer from the model (forward pass).
        # Logits are unnormalized scores for the predicted classes (numbers from 1-10 in this case)
        #   Note: If you wanted actual probabilities (e.g., "73% confident it's a 3"), 
        #         you'd apply softmax to convert logits into a proper probability distribution over the 10 classes.
        # x.shape is (64, 1, 28, 28)
        logits = model(x)

        # check the loss between the model output and the label. F.cross_entropy returns the mean loss over the batch.
        cross_ent_loss: Tensor = F.cross_entropy(input=logits, target=label)
        # print(f'cross_ent_loss: {cross_ent_loss}')

        # Zero out of the (computed) gradients of all tensors because gradients are accumulated into the tensors
        optimizer.zero_grad()

        # Compute the gradients of the loss w.r.t. the weights (leaves of the computation graph) using the chain rule.
        cross_ent_loss.backward()

        # While loss.backward() only computes and stores the gradients in the Tensor's `.grad`, 
        # it doesn't touch the weights of the model. The following line actually does the
        # weight updates, thereby minimizing the loss.
        optimizer.step()

        ## Bookkeeping of loss/accuracy stats

        # print(f'cross_ent_loss: {cross_ent_loss},\
        #       cross_ent_loss.item(): {cross_ent_loss.item()},\
        #       x.size(0): {x.size(0)}')

        # Need to find the total loss of all samples in the batch (x.size(0) should be == BATCH_SIZE), across all mini-batches
        # .item() converts to standard python number
        total_loss += cross_ent_loss.item() * x.size(0)
        total_samples_processed += x.size(0)

        # Get the predictions (numbers from 1-10. logits is the unnormalized score tensor).
        # This is achieved by finding out the index of the maximum value in the logits tensor, which is of shape .
        prediction = logits.argmax(dim=1)
        # print(f'prediction.shape: {prediction.shape}')
        # print(f'preds: {prediction}')

        correct_predictions += (prediction == label).long().sum().item()

    # Return the tuple containing average loss and accuracy
    return (total_loss / total_samples_processed, correct_predictions / total_samples_processed)

# Create the optimizer (with default hyperparameters - learning rate = 0.001). 
# This links the optimizer to the model's parameters
adam_opt = Adam(params=seq_model.parameters())

# Loop over the dataset in batches and train the model
EPOCHS = 3
for epoch in range(0, EPOCHS):
    (avg_loss, avg_accuracy) = train_one_epoch(model=seq_model, optimizer=adam_opt, data_loader=train_loader)
    print(f'avg_loss: {avg_loss}, avg_accuracy: {avg_accuracy}')



/opt/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 938/938 [00:04<00:00, 208.59it/s]


avg_loss: 0.3377004423066974, avg_accuracy: 0.9037166666666666


100%|██████████| 938/938 [00:03<00:00, 272.62it/s]


avg_loss: 0.1365524957406645, avg_accuracy: 0.9586666666666667


100%|██████████| 938/938 [00:03<00:00, 288.28it/s]

avg_loss: 0.09350828526498128, avg_accuracy: 0.9716333333333333


In [8]:
# Model Evaluation

def evaluate_model(model: nn.Sequential, data_loader: DataLoader):
    
    # Set the model in evaluation mode
    model.eval()

    total_samples_processed = 0
    correct_predictions = 0
    total_loss = 0.0

    # Iterate over (mini)batches of data from the test loader
    with torch.no_grad():
        for x, labels in data_loader:

            # move the input tensors to DEVICE
            x, labels = x.to(DEVICE), labels.to(DEVICE)

            # inference
            logits = model(x)

            # check the loss between the model output and the label
            cross_ent_loss: Tensor = F.cross_entropy(input=logits, target=labels)
            # print(f'cross_ent_loss: {cross_ent_loss}')

            # do not train the model (i.e., do not update the weights)
                # optimizer.zero_grad()? no - don't zero out the gradients
                # loss.backward()? no - don't backprop the gradients
                # optimizer.step() no - don't update the weights

            # report the average loss and accuracy back
            total_loss += cross_ent_loss.item() * x.size(0)
            predictions = logits.argmax(dim=1)
            correct_predictions += (predictions == labels).long().sum().item()
            total_samples_processed += x.size(0)

    avg_loss = total_loss / total_samples_processed
    avg_accuracy = correct_predictions / total_samples_processed

    return (avg_loss , avg_accuracy)

evaluate_model(model=seq_model, data_loader=test_loader)

(0.09751689592693001, 0.9693)

In [9]:
# Export the model
def export_model(model):
    torch.save(model.state_dict(), "mnist_cnn.pth")

export_model(seq_model)
